# This nb is used to generate the calibration files for the different ABCs' heaters

## Imports

In [ ]:
import pandas as pd
from InfluxDBInterface.libdb import download_data_DB
from libcal import getUUIDFromBoardID, frameBasePower, findMaxHtrPowers, writeCalibration

## Configuration

In [2]:
board_id = "abc02"

## Main code

In [ ]:
first_dt = pd.Timestamp("2025-12-01T10:00:00Z")
last_dt = pd.Timestamp.now(tz=pd.Timestamp.utcnow().tz)
res = 3 # resolution in seconds.
bucket = "a_sensing"

mcu_uuid = getUUIDFromBoardID(board_id)
if mcu_uuid is None:
    raise ValueError(f"Board ID {board_id} not found in the database.")

filters = {
    "measurement": ["pwr", "htr"],
    "mcu_uuid": mcu_uuid
}

data = download_data_DB(bucket, first_dt, last_dt, res, filters)
print(f"Found {len(data)} entries between {first_dt} and {last_dt} for board ID {board_id}.")

base_power = frameBasePower(data)
print(f"Base power consumption: {base_power} W")

max_htr_powers = findMaxHtrPowers(data, base_power)
print(f"Max heater powers (W) for {board_id}:")
print(max_htr_powers)

# Persist the new calibration into RHCConfigs/<rhc-name>.cfg, preserving the mcu_uuid /
# exclude_sensors already recorded for this robot.
writeCalibration(board_id, max_htr_powers, base_power)